<a href="https://colab.research.google.com/github/iMaIrsdyh/BigData26_B_2411533018_KarimahIrsyadiyah/blob/main/Praktikum2/BD_B_P02_2411533018_KarimahIrsyadiyah.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# KARIMAH IRSYADIYAH
**2411533018**

**PRAKTIKUM 2 BIG DATA**

In [43]:
!pip install faker -q

In [44]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

import os

DIR_KERJA = "/content/data"
DIR_SIMPAN = "/content/drive/MyDrive/BigData/Praktikum2"

os.makedirs(DIR_KERJA, exist_ok=True)
os.makedirs(DIR_SIMPAN, exist_ok=True)

print(os.listdir(DIR_SIMPAN))

Mounted at /content/drive
[]


# K. Langkah Kerja

**K-1. Import Library dan Inisialisasi**

In [45]:
import numpy as np
import pandas as pd
from faker import Faker
import random

Library yang digunakan diinisialisasi pada tahap awal praktikum. NumPy digunakan untuk operasi numerik dan pembuatan data acak, sedangkan Pandas digunakan untuk mengolah data dalam bentuk DataFrame. Faker digunakan untuk menghasilkan data sintetis seperti nama dan informasi lainnya. Library random digunakan untuk menghasilkan nilai acak dalam proses pembuatan data.

**K-2. Membuat Dataset Sintetis (Simulasi Data Acquisition)**

In [46]:
SEED = 7
np.random.seed(SEED)
random.seed(SEED)
fake = Faker("id_ID")
Faker.seed(SEED)

N = 500
kategori_produk = ["Elektronik", "Fashion", "Kesehatan", "Rumah Tangga", "Olahraga", "Buku"]
metode_bayar = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]

rows = []
for i in range(1, N + 1):
    trx_id = f"TRX{i:05d}"
    nama_pelanggan = fake.name()
    produk = fake.word().capitalize() + " " + random.choice(["Pro", "Lite", "Max", "Basic", "Mini"])
    kategori = random.choice(kategori_produk)
    harga_dasar = random.choice([15000, 25000, 50000, 75000, 120000, 250000, 500000, 1200000])
    qty = random.randint(1, 5)

    # Variasi format harga: angka polos, ada "Rp", ada desimal ".0", ada spasi
    harga_variants = [
        str(harga_dasar),
        f"Rp{harga_dasar:,}".replace(",", "."),
        f"{harga_dasar}.0",
        f" {harga_dasar} "
    ]
    harga = random.choice(harga_variants)

    # Variasi format tanggal: ISO, DD/MM/YYYY, DD-MM-YYYY
    tgl = fake.date_between(start_date="-90d", end_date="today")
    tgl_variants = [tgl.strftime("%Y-%m-%d"), tgl.strftime("%d/%m/%Y"),
                    tgl.strftime("%d-%m-%Y")]
    tanggal = random.choice(tgl_variants)

    metode = random.choice(metode_bayar)
    if random.random() < 0.3:
        metode = metode.lower()
    if random.random() < 0.2:
        kategori = kategori.upper() + " "

    kota = fake.city()
    rating = random.choice([1, 2, 3, 4, 5, None, None])  # rating opsional

    rows.append({
        "transaction_id": trx_id, "customer_name": nama_pelanggan, "product_name": produk.strip(),
        "category": kategori, "price": harga, "quantity": qty, "payment_method": metode,
        "transaction_date": tanggal, "shipping_city": kota, "rating": rating,
    })

membuat dataset transaksi sintetis sebanyak 500 data. Data dibuat menggunakan Faker dan nilai acak dengan SEED = 42 agar hasil yang diperoleh tetap sama setiap kali kode dijalankan. Pada data sengaja dibuat beberapa variasi format harga dan tanggal, penggunaan huruf besar atau kecil, missing value, serta rating yang bersifat opsional. Variasi tersebut digunakan untuk mensimulasikan kondisi data mentah yang dapat ditemukan dalam proses acquisition.

In [47]:
df = pd.DataFrame(rows)

# Suntikkan missing value pada beberapa kolom
for col, frac in [("customer_name", 0.02), ("shipping_city", 0.03), ("payment_method", 0.015)]:
    idx = df.sample(frac=frac, random_state=SEED).index
    df.loc[idx, col] = np.nan

Data transaksi yang telah dibuat dimasukkan ke dalam DataFrame menggunakan Pandas. Setelah itu, missing value sengaja ditambahkan pada kolom customer_name, shipping_city, dan payment_method dengan persentase tertentu. Langkah ini mensimulasikan data mentah yang tidak lengkap dan nantinya dapat digunakan pada tahap prapemrosesan.

In [48]:
# Duplikasi 15 baris (mensimulasikan transaksi yang tercatat dua kali)
dup_rows = df.sample(n=15, random_state=SEED)
df = pd.concat([df, dup_rows], ignore_index=True)
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

df.to_csv("transaksi_mentah.csv", index=False)
print("Jumlah baris:", len(df))

Jumlah baris: 515


Sebanyak 15 baris data diduplikasi untuk mensimulasikan transaksi yang tercatat lebih dari satu kali. Data kemudian diacak kembali dan disimpan dalam file transaksi_mentah.csv. Setelah penambahan duplikat, jumlah data menjadi 515 baris.

**K-3. Deteksi dan Penanganan Missing Value**

In [49]:
print(df.isnull().sum())

transaction_id        0
customer_name        20
product_name          0
category              0
price                 0
quantity              0
payment_method       16
transaction_date      0
shipping_city        30
rating              120
dtype: int64


In [50]:
df = df.dropna(subset=["customer_name", "payment_method"])
df["shipping_city"] = df["shipping_city"].fillna("Tidak Diketahui")

Missing value pada customer_name dan payment_method ditangani dengan menghapus baris karena kedua informasi tersebut dianggap penting dalam data transaksi. Missing value pada shipping_city diganti dengan "Tidak Diketahui" agar data transaksi tetap dapat digunakan untuk analisis lainnya. Sementara itu, kolom rating tetap dibiarkan sebagai NaN karena rating bersifat opsional.

**K-4. Deteksi dan Penanganan Duplicate**

In [51]:
print("Baris duplicate (semua kolom sama):", df.duplicated().sum())
print("transaction_id duplicate:", df["transaction_id"].duplicated().sum())

df = df.drop_duplicates()
print("Jumlah baris setelah drop_duplicates():", len(df))

Baris duplicate (semua kolom sama): 5
transaction_id duplicate: 5
Jumlah baris setelah drop_duplicates(): 490


mendeteksi data yang tercatat lebih dari satu kali. Pemeriksaan dilakukan pada seluruh kolom dan pada transaction_id untuk melihat adanya transaksi yang terduplikasi. Data duplicate kemudian dihapus menggunakan drop_duplicates() agar setiap transaksi hanya tercatat satu kali.

**K-5. Koreksi Tipe Data dan Standardisasi Format**

K-5a. Standardisasi teks kategorikal

In [52]:
for col in ["category", "payment_method", "shipping_city"]:
    df[col] = df[col].astype("string").str.strip().str.title()

# "Cod" akan dianggap salah; kembalikan ke huruf kapital sesuai standar
df["payment_method"] = df["payment_method"].replace({"Cod": "COD"})

menstandarkan format teks pada kolom category, payment_method, dan shipping_city. Spasi yang tidak diperlukan dihapus menggunakan str.strip(), kemudian format huruf diseragamkan menggunakan str.title(). Nilai Cod pada metode pembayaran dikembalikan menjadi COD agar sesuai dengan format standar.

K-5b. Koreksi tipe data pada kolom price

In [53]:
def bersihkan_harga(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().replace("Rp", "").replace(".", "").replace(",", ".")
    try:
        return float(x)
    except ValueError:
        return np.nan

df["price"] = df["price"].apply(bersihkan_harga)

Kolom price memiliki beberapa variasi format seperti penggunaan Rp, tanda titik, koma, dan spasi. Fungsi bersihkan_harga() digunakan untuk menghapus simbol dan menyesuaikan format angka agar dapat dikonversi menjadi tipe numerik. Nilai yang tidak dapat dikonversi akan diubah menjadi NaN.

K-5c. Standardisasi format tanggal ke YYYY-MM-DD

In [54]:
def parse_tanggal(x):
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y"):
        try:
            return pd.to_datetime(x, format=fmt)
        except:
            continue
    return pd.NaT

df["transaction_date"] = df["transaction_date"].apply(parse_tanggal).dt.strftime("%Y-%m-%d")

Kolom transaction_date memiliki beberapa format tanggal, yaitu YYYY-MM-DD, DD/MM/YYYY, dan DD-MM-YYYY. Fungsi parse_tanggal() mencoba membaca tanggal berdasarkan format yang telah ditentukan. Setelah berhasil dikonversi, seluruh tanggal diseragamkan menjadi format YYYY-MM-DD.

K-5d. Finalisasi tipe data

In [55]:
df["quantity"] = df["quantity"].astype(int)
df["price"] = df["price"].astype(float)

Kolom quantity dikonversi menjadi tipe integer karena berisi jumlah barang, sedangkan price dikonversi menjadi tipe float karena berisi nilai harga yang dapat memiliki angka desimal.

**K-6. Ekspor Dataset Bersih**

In [56]:
df.to_csv("transaksi_bersih.csv", index=False)
print("Dataset bersih tersimpan:", len(df), "baris")

Dataset bersih tersimpan: 490 baris


Dataset yang telah melalui proses pembersihan dan prapemrosesan disimpan dalam format CSV dengan nama transaksi_bersih.csv.

# STUDI KASUS

**1. Mengapa jumlah transaksi 515 dan 490 bisa berbeda?**

Perbedaan terjadi karena 515 baris merupakan data setelah simulasi penambahan duplicate, sedangkan 490 baris merupakan data setelah melalui proses prapemrosesan.

Pada K-2, dataset awal berjumlah 500 baris, kemudian ditambahkan 15 baris duplikat sehingga menjadi 515 baris. Setelah itu dilakukan penanganan missing value. Baris yang memiliki missing value pada customer_name dan payment_method dihapus karena kedua kolom tersebut dianggap penting untuk transaksi. Setelah proses tersebut, sebagian duplicate ikut terhapus sehingga tersisa 495 baris.

Selanjutnya pada K-4 ditemukan 5 baris duplicate, kemudian dihapus menggunakan drop_duplicates(). Hasil akhirnya menjadi 490 baris.

Jadi:

515 → penanganan missing value → 495 → hapus 5 duplicate → 490 baris.

**2. Apakah 490 baris “lebih benar” dibandingkan 515 baris?**

Tidak bisa langsung dikatakan bahwa 490 selalu lebih benar. Namun, 490 merupakan dataset yang sudah melalui proses pemeriksaan dan pembersihan, sedangkan 515 masih mengandung masalah kualitas data.

Hal ini berkaitan dengan konsep Veracity, yaitu tingkat kebenaran, keakuratan, dan keandalan data. Pada proses preprocessing ditemukan missing value dan duplicate yang dapat memengaruhi hasil analisis. Karena itu, data perlu diperiksa dan dibersihkan sebelum digunakan.

Dalam kasus ini, 490 baris lebih layak digunakan untuk analisis setelah preprocessing, karena duplicate telah ditangani dan missing value pada kolom yang dianggap penting telah ditangani. Tetapi penghapusan data juga harus memiliki alasan yang jelas agar tidak menghilangkan informasi yang sebenarnya masih berguna.

**3. Mengapa rating tetap memiliki missing value?**

Kolom rating sengaja dibiarkan memiliki NaN karena rating bersifat opsional. Tidak semua pembeli wajib memberikan rating setelah melakukan transaksi. Oleh karena itu, nilai kosong pada rating tidak otomatis berarti datanya salah.

Jika Finance ingin mengetahui rata-rata rating, perhitungannya sebaiknya dilakukan hanya pada transaksi yang memang memiliki rating.

# Q. Latihan

**Q1. Mengubah SEED menjadi 7**

| SEED | `transaksi_mentah.csv` | `transaksi_bersih.csv` |
| ---- | ---------------------: | ---------------------: |
| 42   |                    515 |                    490 |
| 7    |                    515 |                    490 |


Jumlah baris pada kedua SEED sama, yaitu 515 baris sebelum preprocessing dan 490 baris setelah preprocessing. Perbedaan SEED memengaruhi data acak yang terbentuk, tetapi tidak mengubah jumlah baris karena struktur proses pembuatannya tetap sama.

**Q2. Pemeriksaan Harga Tidak Valid**

In [57]:
df["is_valid_price"] = df["price"] > 0

print(df["is_valid_price"].value_counts())
print("Jumlah harga tidak valid:", (~df["is_valid_price"]).sum())

is_valid_price
True    490
Name: count, dtype: int64
Jumlah harga tidak valid: 0


Kolom is_valid_price digunakan untuk memeriksa validitas harga. Nilai akan True jika price lebih besar dari 0 dan False jika harga kurang dari atau sama dengan 0. Hasil pemeriksaan menunjukkan seluruh 490 transaksi memiliki harga yang valid, sehingga tidak ditemukan harga yang kurang dari atau sama dengan 0.

**Q3. Jumlah Transaksi per Category**

In [58]:
df["category"].value_counts()

,count
category,
Rumah Tangga,91
Kesehatan,86
Buku,81
Fashion,80
Elektronik,78
Olahraga,74


value_counts() digunakan untuk menghitung jumlah transaksi pada setiap kategori produk. Hasil menunjukkan bahwa kategori Rumah Tangga memiliki jumlah transaksi terbanyak, sedangkan Olahraga memiliki jumlah transaksi paling sedikit pada dataset bersih dengan SEED = 7.